<a href="https://colab.research.google.com/github/neha-lokireddy/BDA-Assignment/blob/main/bda_assignment_q3__083.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation Engine with Apache Spark

First, let's install `pyspark` which is the Python API for Spark. This allows us to interact with Spark programmatically.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install pyspark

Next, we will download the MovieLens 100K dataset, which is a popular dataset for recommendation systems. We'll extract it to a local directory.

In [ ]:
!wget http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip ml-100k.zip -d ml-100k

Now, let's initialize a SparkSession. This is the entry point to programming Spark with the Dataset and DataFrame API.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieRecommendation") \
    .getOrCreate()

print("SparkSession created successfully!")

We'll load the `u.data` file, which contains user ratings. This file has four columns: `user id | item id | rating | timestamp`. We're interested in the first three. We'll also cast them to appropriate types for our ALS model.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, FloatType

# Define the schema for the ratings data
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("itemId", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", LongType(), True)
])

# Load the ratings data
ratings_df = spark.read.csv(
    "ml-100k/ml-100k/u.data",
    sep="\t",
    schema=ratings_schema
)

# Select only the relevant columns for ALS and cache for performance
ratings = ratings_df.select("userId", "itemId", "rating").cache()

print("Ratings data loaded:")
ratings.show(5)
ratings.printSchema()

Now, we'll split the data into training and test sets to evaluate our model's performance. Then, we'll build and train an Alternating Least Squares (ALS) model, which is commonly used for collaborative filtering in Spark.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

# Split the data into training and test sets
(training, test) = ratings.randomSplit([0.8, 0.2], seed=42)

# Build the recommendation model using ALS
als = ALS(
    maxIter=5, # Number of iterations
    regParam=0.01, # Regularization parameter
    userCol="userId",
    itemCol="itemId",
    ratingCol="rating",
    coldStartStrategy="drop", # Strategy for dealing with new users/items (cold start)
    seed=42
)

# Train the model on the training data
model = als.fit(training)

print("ALS model trained successfully!")

Let's evaluate the model using the test set. We'll use Root Mean Squared Error (RMSE) to measure how close our predicted ratings are to the actual ratings.

In [ ]:
# Make predictions on the test set
predictions = model.transform(test)

# Evaluate the model by computing the RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)

print(f"Root-mean-square error = {rmse}")

# Show some predictions
print("Sample predictions:")
predictions.show(5)

Finally, let's generate some movie recommendations for a few users. We can recommend the top 10 movies for all users or specific users.

In [ ]:
# Generate top 10 movie recommendations for each user
userRecs = model.recommendForAllUsers(10)

print("Top 10 movie recommendations for all users:")
userRecs.show(5, truncate=False)

# Let's pick a specific user, e.g., user ID 1
# To make these recommendations more readable, we'd typically join with a movie metadata DataFrame.
# For now, let's just display the recommendations for one user.

# Filter recommendations for a specific user (e.g., userId = 1)
specific_user_id = 1
specific_user_recs = userRecs.filter(userRecs.userId == specific_user_id)

print(f"\nTop recommendations for user {specific_user_id}:")
specific_user_recs.show(truncate=False)

spark.stop()
print("SparkSession stopped.")